# Cleaning Data
Starts from the raw data saved by `01_pulling_data`. Run that notebook first. Values are shown as raw JSON, as in `01_pulling_data`.

All times are standardized to UTC first.

In [22]:
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def load(endpoint):
    data = json.loads(Path(f"data/raw/{endpoint}.json").read_text())
    # The API wraps the records in an extra list: [[{...}, {...}]]
    return [record for page in data for record in page]


def show(records):
    # Show each value as its raw JSON so types, nulls and stray spaces stay visible.
    table = pd.DataFrame(
        [{key: json.dumps(value) for key, value in record.items()} for record in records]
    ).fillna("")
    left = [("font-family", "monospace"), ("text-align", "left"), ("white-space", "pre")]
    return (
        table.style.hide(axis="index")
        .set_caption(f"{len(records)} records")
        .set_table_styles([{"selector": "th, td", "props": left}])
    )


production_runs = load("production_runs")
counter_readings = load("counter_readings")
downtime_events = load("downtime_events")

In [23]:
def to_utc(timestamp):
    # Convert an ISO timestamp to UTC written with Z, e.g. "T08:00:00-05:00" -> "T13:00:00Z"
    if timestamp is None:
        return None  # Keep null end times (still in progress)
    parsed = datetime.fromisoformat(timestamp)
    # Refuse to guess a time zone the timestamp doesn't state
    if parsed.tzinfo is None:
        raise ValueError(f"Timestamp has no time zone: {timestamp!r}")
    return parsed.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


production_runs = [
    {**run, "start_time": to_utc(run["start_time"]), "end_time": to_utc(run["end_time"])}
    for run in production_runs
]
counter_readings = [
    {**reading, "recorded_at": to_utc(reading["recorded_at"])}
    for reading in counter_readings
]
downtime_events = [
    {**event, "start_time": to_utc(event["start_time"]), "end_time": to_utc(event["end_time"])}
    for event in downtime_events
]

## Production runs

In [24]:
# 1. Trim and uppercase product_code, e.g. " flour-a " -> "FLOUR-A"
production_runs = [
    {**run, "product_code": run["product_code"].strip().upper()} for run in production_runs
]

# 2. Convert target_rate to cases/hour; an unknown unit raises a KeyError instead of guessing
CASES_PER_HOUR = {"cases/hour": 1, "cases/minute": 60}
production_runs = [
    {
        **run,
        "target_rate": run["target_rate"] * CASES_PER_HOUR[run["target_rate_unit"]],
        "target_rate_unit": "cases/hour",
    }
    for run in production_runs
]

# 3. Replace target_rate and target_rate_unit with target_hourly_rate and unit
production_runs = [
    {
        **{key: value for key, value in run.items() if key not in ("target_rate", "target_rate_unit")},
        "target_hourly_rate": run["target_rate"],
        "unit": run["target_rate_unit"].split("/")[0],
    }
    for run in production_runs
]

show(production_runs)

run_id,equipment_id,start_time,end_time,product_code,target_hourly_rate,unit
"""R100""","""PKG-01""","""2026-08-13T13:00:00Z""","""2026-08-13T14:00:00Z""","""FLOUR-A""",600,"""cases"""
"""R101""","""PKG-01""","""2026-08-13T14:10:00Z""","""2026-08-13T15:10:00Z""","""FLOUR-A""",600,"""cases"""
"""R102""","""PKG-02""","""2026-08-13T13:15:00Z""","""2026-08-13T14:15:00Z""","""FLOUR-B""",420,"""cases"""
"""R103""","""PKG-02""","""2026-08-13T14:30:00Z""","""2026-08-13T15:30:00Z""","""FLOUR-B""",420,"""cases"""
"""R104""","""PKG-01""","""2026-08-13T15:20:00Z""",null,"""FLOUR-C""",540,"""cases"""


## Counter readings

In [25]:
# 1. Split payload into count and unit, values unchanged
counter_readings = [
    {
        **{key: value for key, value in reading.items() if key != "payload"},
        "count": reading["payload"]["count"],
        "unit": reading["payload"]["unit"],
    }
    for reading in counter_readings
]


# 2. Convert count to integers: numeric text such as "1180" becomes 1180; anything else stays as is
def to_int(count):
    if isinstance(count, str) and count.strip().isdecimal():
        return int(count)
    return count


counter_readings = [{**reading, "count": to_int(reading["count"])} for reading in counter_readings]

show(counter_readings)

reading_id,equipment_id,recorded_at,count,unit
"""C001""","""PKG-01""","""2026-08-13T12:59:00Z""",1000,"""cases"""
"""C002""","""PKG-01""","""2026-08-13T13:10:00Z""",1090,"""cases"""
"""C003""","""PKG-01""","""2026-08-13T13:20:00Z""",1180,"""cases"""
"""C004""","""PKG-01""","""2026-08-13T13:20:00Z""",1180,"""cases"""
"""C005""","""PKG-01""","""2026-08-13T13:40:00Z""","""ERROR""","""cases"""
"""C006""","""PKG-01""","""2026-08-13T14:00:00Z""",1510,"""cases"""
"""C007""","""PKG-01""","""2026-08-13T14:10:00Z""",1510,"""cases"""
"""C008""","""PKG-01""","""2026-08-13T14:30:00Z""",1680,"""cases"""
"""C009""","""PKG-01""","""2026-08-13T14:40:00Z""",1760,"""cases"""
"""C010""","""PKG-01""","""2026-08-13T14:50:00Z""",25,"""cases"""


## Downtime events

In [26]:
show(downtime_events)

event_id,equipment_id,start_time,end_time,reason
"""D001""","""PKG-01""","""2026-08-13T13:30:00Z""","""2026-08-13T13:36:00Z""","""Material unavailable"""
"""D002""","""PKG-01""","""2026-08-13T14:08:00Z""","""2026-08-13T14:14:00Z""","""Unknown"""
"""D003""","""PKG-01""","""2026-08-13T14:43:30Z""","""2026-08-13T14:47:30Z""","""material"""
"""D004""","""PKG-01""","""2026-08-13T14:46:00Z""","""2026-08-13T14:49:00Z""","""MATL"""
"""D005""","""PKG-02""","""2026-08-13T13:40:00Z""","""2026-08-13T13:45:00Z""","""Jam"""
"""D006""","""PKG-02""","""2026-08-13T14:48:00Z""","""2026-08-13T14:49:00Z""","""Unknown"""
"""D007""","""PKG-02""","""2026-08-13T14:53:20Z""","""2026-08-13T14:54:10Z""","""Unknown"""
"""D008""","""PKG-02""","""2026-08-13T14:58:05Z""","""2026-08-13T14:59:00Z""","""Unknown"""
"""D009""","""PKG-02""","""2026-08-13T15:03:40Z""","""2026-08-13T15:04:30Z""","""Unknown"""
"""D010""","""PKG-01""","""2026-08-13T15:34:00Z""",null,"""Operator stop"""
